# Lab 10/Assignment 4 - Image Captioning using Transformers
For this assignemnt, you will be given full-colour images as input data, and attempt to develop an Encoder-Decoder model to create worded captions for the image. For instance, for the image 

<img src='https://towardsdatascience.com/wp-content/uploads/2022/06/0HxBDFFqwpjShEnyb-scaled.jpg' width='250px'/>

Your model may generate something like "A dog running in the sea"


We will work with the Flickr8k dataset, which contains 8000 images, each associated with 5 sample captions.

## Flickr8k Dataset

This dataset contains approximately 1GB of data. Therefore it is strongly reccommended to use the UCC Jupyter server for this assignment.

In [1]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

In [2]:
!mkdir -p data/flickr8k/
!wget "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip" -O "data/flickr8k/Flickr8k_Dataset.zip"
!wget "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip" -O "data/flickr8k/Flickr8k_text.zip"

The syntax of the command is incorrect.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
%%bash
if [ ! -d "data/flickr8k/Flicker8k_Dataset" ]
then
    unzip "data/flickr8k/Flickr8k_Dataset.zip" -d data/flickr8k/
fi

if [ ! -d "data/flickr8k/Flickr8k_text" ]
then
    unzip "data/flickr8k/Flickr8k_text.zip" -d data/flickr8k/Flickr8k_text
    rm -r "data/flickr8k/Flickr8k_text/__MACOSX"
fi

if [ -d "data/flickr8k/__MACOSX" ]
then
    rm -r "data/flickr8k/__MACOSX"
fi
mkdir -p saved_models

Couldn't find program: 'bash'


## Data Loading

In [4]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import os
import numpy as np

from collections import Counter

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader,Dataset
import torchvision.transforms as T

from PIL import Image


You may ned to use `%cd __wherever__` here`

In [ ]:
data_location =  "data/flickr8k"

caption_file = data_location + '/Flickr8k_text/Flickr8k.token.txt'
df = pd.read_csv(caption_file, delimiter='\t', header=None)

# Cleanup filenames
df.iloc[:,0] = df[0].apply(lambda x: x.split('#')[0])
df = df[~df[0].apply(lambda x: '.jpg.1' in x)]

print("There are {} unique captions".format(len(df)))


In [ ]:
data_idx = 42

image_path = data_location+"/Flicker8k_Dataset/"+df.iloc[data_idx,0]
img=mpimg.imread(image_path)
plt.imshow(img)
plt.show()

all_captions=df[1][df[0]==df.iloc[data_idx,0]]

for cap in all_captions:
    print("Caption:",cap)

## Creating PyTorch Dataset

### Tokenization

The first step is to splitthe captions into uniformly formatted lists of 'tokens', which act as the inputs to our models. Tokens can be words or punctuation, which are used similarly as how we used characters in the previous lab.

In order to clean up the text, we will use the Python SpaCY NLP module

In [ ]:
%pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

In [ ]:
spacy_eng = spacy.load("en_core_web_sm")

We will use this to split sentences into "tokens", which we will pass as inputs to our models.

In [ ]:
text = "This is a good place - to find a city!"
[token.text.lower() for token in spacy_eng.tokenizer(text)]

### Vocabulary

We will map every possible 'token' that we encounter into a distinct integer.

In [ ]:
class Vocabulary:
    def __init__(self,freq_threshold):
        #setting the pre-reserved tokens int to string tokens
        self.itos = {0:"<PAD>",1:"<SOS>",2:"<EOS>",3:"<UNK>"}
        
        #string to int tokens
        #its reverse dict self.itos
        self.stoi = {v:k for k,v in self.itos.items()}
        
        self.freq_threshold = freq_threshold
        
    def __len__(self): return len(self.itos)
    
    @staticmethod
    def tokenize(text):
        return [token.text.lower() for token in spacy_eng.tokenizer(text)]
    
    def build_vocab(self, sentence_list):
        frequencies = Counter()
        idx = 4
        
        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1
                
                #add the word to the vocab if it reaches minum frequecy threshold
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self,text):
        """ For each word in the text corresponding index token for that word form the vocab built as list """
        tokenized_text = self.tokenize(text)
        return [ self.stoi[token] if token in self.stoi else self.stoi["<UNK>"] for token in tokenized_text ] 

In [ ]:
v = Vocabulary(freq_threshold=1)

v.build_vocab(["This is a good place - to find a city!"])
print(v.stoi)
print(v.numericalize("This is a good place ; to find a city here!"))

Notice how ';' and 'here' are unknown tokens, which our volcabularly maps to \<UNK\> -- unkown. \<SOS\> and \<EOS\> Correpsond to Start and End of Sentence

### Dataset

There are ~40000 samples. We will use 30000 samples for training, and the remainder for validation.

In [ ]:
class FlickrDataset(Dataset):
    """
    FlickrDataset
    """
    def __init__(self,root_dir,captions_file,transform=None,train=True,freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(caption_file, delimiter='\t', header=None)
        # Cleanup filenames
        np.random.seed(42)
        self.df = self.df.sample(frac=1)
        
        self.df =  self.df.reset_index(drop=True)
        self.df.iloc[:,0] = self.df[0].apply(lambda x: x.split('#')[0])
        self.df = self.df[~self.df[0].apply(lambda x: '.jpg.1' in x)]
        self.df =self.df.reset_index()
        self.transform = transform
        
        #Get image and caption colum from the dataframe
        self.imgs = self.df[0]
        self.captions = self.df[1]
        
        #Initialize vocabulary and build vocab
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocab(self.captions.tolist())
        
        if train:
            self.df = self.df.iloc[:30000,:]
        else:
            self.df = self.df.iloc[30000:,:]
        
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        caption = self.captions[idx]
        img_name = self.imgs[idx]
        img_location = os.path.join(self.root_dir,img_name)
        img = Image.open(img_location).convert("RGB")
        
        #apply the transfromation to the image
        if self.transform is not None:
            img = self.transform(img)
        
        #numericalize the caption text
        caption_vec = []
        caption_vec += [self.vocab.stoi["<SOS>"]]
        caption_vec += self.vocab.numericalize(caption)
        caption_vec += [self.vocab.stoi["<EOS>"]]
        
        return img, torch.tensor(caption_vec)

In [ ]:
# Downscale the images, convert to Tensor
transforms = T.Compose([
    T.Resize((224,224)),
    # These are the manually computed means and stdevs in each dimension
    T.ToTensor(),
    T.Normalize((0.485, 0.4461, 0.4039), (0.2695, 0.2623, 0.2772)),
    
])

In [ ]:
train_dataset =  FlickrDataset(
    root_dir = data_location+"/Flicker8k_Dataset",
    captions_file = data_location+"/Flickr8k_text/Flickr8k.token.txt",
    transform=transforms
)

val_dataset =  FlickrDataset(
    root_dir = data_location+"/Flicker8k_Dataset",
    captions_file = data_location+"/Flickr8k_text/Flickr8k.token.txt",
    train=False,
    transform=transforms
)

In [ ]:
def show_image(inp, title=None):
    """Imshow for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001) 

In [ ]:
img, caps = train_dataset[0]
show_image(img,"Image")
print("Token:",caps)
print("Sentence:")
print([train_dataset.vocab.itos[token] for token in caps.tolist()])


### Dataloader

In oder to enable batching for our dataloader, we must pad our captions with dummy tokens to be of uniform length:

In [ ]:
class PadCaptions:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets
    
pad_idx = train_dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    collate_fn = PadCaptions(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    collate_fn = PadCaptions(pad_idx=pad_idx)
)

# Part 1 - Implementing Scaled Dot Product Attention

<img src='https://agrimpaneru.com.np/blog/self-attention-pytorch/image.png' width='550px'/>

Your task is to manually implement scaled dot produce attention, and apply it to a sample sentence.

A prerequisite step here is to create an *embedding* from this tokenised sentence



## Dataset

We will modify our dataset here, to contain only raw text, without tokenization, and discards the images.

In [ ]:
class FlickrDatasetSimple(Dataset):
    """
    FlickrDataset
    """
    def __init__(self,root_dir,captions_file,transform=None,train=True,freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(caption_file, delimiter='\t', header=None)
        # Cleanup filenames
        np.random.seed(42)
        self.df = self.df.sample(frac=1)
        
        self.df =  self.df.reset_index(drop=True)
        self.df.iloc[:,0] = self.df[0].apply(lambda x: x.split('#')[0])
        self.df = self.df[~self.df[0].apply(lambda x: '.jpg.1' in x)]
        self.df =self.df.reset_index()
        self.transform = transform
        
        #Get image and caption colum from the dataframe
        self.imgs = self.df[0]
        self.captions = self.df[1]
            
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        caption = self.captions[idx]
        
        #numericalize the caption text
        caption_vec = [caption]
        
        return caption_vec

In [ ]:
train_simple_dataset =  FlickrDatasetSimple(
    root_dir = data_location+"/Flicker8k_Dataset",
    captions_file = data_location+"/Flickr8k_text/Flickr8k.token.txt",
)


## Embeddings

Ww will us a pretrained Embedding layer (model) here, instead of trinaing one from scratch. We will use the well-known Word2Vec model, which is worth reading more about. We will load a particular pre-trained Word2Vec model that was trained on a large number of Twitter posts

In [ ]:
%pip install gensim

In [ ]:
from gensim.models import Word2Vec
import gensim.downloader as api

w2v_model = api.load("glove-twitter-25")

In [ ]:
weights = torch.FloatTensor(w2v_model.vectors)
embedding = torch.nn.Embedding.from_pretrained(weights)

As a sidebar, there are all sorts of interesting operations you can do over the Word2Vec model:

In [ ]:
w2v_model.most_similar("computer",topn=10)

In [ ]:
w2v_model.most_similar("summer",topn=10)

In [ ]:
w2v_model.doesnt_match(["dog","cat","elephant","orange","kangaroo"])

The main feature here is that we can now use this very effective embeding model to generate embeddings of each word in our dataset.

Consider the process for the word "dog"

In [ ]:
word = 'dog'

### Get the index of each word in the Word2Vec vocuabulary

In [ ]:
idx = w2v_model.key_to_index[word.lower()]

### Convert to tensor

In [ ]:
input_t = torch.LongTensor([idx])

### Get the Embedding

In [ ]:
embedding(input_t)

## Task 1:

1. You are given a string, the first sentence in the dataset
2. Create a matrix, where the number of rows corresponds to the number of words in the sentence, and has 25 columns (the embedding size)
3. Populate the matrix with the embedding for the sentence

In [ ]:
sentence = train_simple_dataset[0][0].split()

In [ ]:
# EMBED THE SENTENCE INTO A MATRIX named `sentence_embedding`

# Build list of indices for every word in the sentence (skip OOV words)
indices = []
for word in sentence:
    w = word.lower()
    if w in w2v_model.key_to_index:
        indices.append(w2v_model.key_to_index[w])
    else:
        # Words not in the Word2Vec vocabulary are skipped
        print(f"Skipping OOV word: '{word}'")

# Convert to a LongTensor of shape (num_words,)
indices_tensor = torch.LongTensor(indices)

# Pass through the Embedding layer -> shape (num_words, 25)
sentence_embedding = embedding(indices_tensor)

print(f"Sentence: {sentence}")
print(f"Embedding matrix shape: {sentence_embedding.shape}")
sentence_embedding


## Task 2: Implement Scaled Dot Product Attention    

In [ ]:
def scaled_dot_product_attention(query, key, value):
    """
    Compute Scaled Dot Product Attention.
    Formula: softmax(QK^T / sqrt(d_k)) V

    Args:
        query : (seq_len, d_k)
        key   : (seq_len, d_k)
        value : (seq_len, d_v)

    Returns:
        output  : (seq_len, d_v)  – weighted combination of values
        weights : (seq_len, seq_len) – attention weight matrix
    """
    import torch.nn.functional as F
    import math

    # Compute raw dot-product scores and scale by sqrt(d_k)
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Apply Softmax over the key dimension to get attention weights
    weights = F.softmax(scores, dim=-1)   # (seq_len, seq_len)

    # Weighted sum of values
    output = torch.matmul(weights, value) # (seq_len, d_v)

    return output, weights


In [ ]:
sentence = train_simple_dataset[0][0].split()

d_model = 25

indices = []
valid_words = []
for word in sentence:
    w = word.lower()
    if w in w2v_model.key_to_index:
        indices.append(w2v_model.key_to_index[w])
        valid_words.append(word)

indices_tensor = torch.LongTensor(indices)
embeddings = embedding(indices_tensor)   # shape: (seq_len, 25)

# Apply self-attention (Q = K = V = embeddings)
q = k = v = embeddings
output, weights = scaled_dot_product_attention(q, k, v)

print(f"Input embedding shape : {embeddings.shape}")
print(f"Attention output shape: {output.shape}")
print(f"Attention weight shape: {weights.shape}")

# Visualization
plt.figure(figsize=(10, 8))
plt.imshow(weights.detach().numpy(), cmap='viridis')
plt.xticks(range(len(valid_words)), valid_words, rotation=45, ha='right')
plt.yticks(range(len(valid_words)), valid_words)
plt.title("Self-Attention Heatmap")
plt.colorbar()
plt.tight_layout()
plt.show()


# Part 2: A Transformer-based Image Captioning Model

The following is an extension of the previous assignment, with the only difference being in the Decoder architecture!

In [ ]:
import torch.nn as nn

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        # Layer 1: Input (3, 224, 224) -> Output (32, 112, 112)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Layer 2: Output (64, 56, 56)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Layer 3: Output (128, 28, 28)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Fully connected layer to reach embed_size
        # Assuming input image is 224x224, after 3 pools it is 28x28
        self.fc = nn.Linear(128 * 28 * 28, embed_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        x = self.pool1(F.relu(self.conv1(images)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        
        # Flatten the features
        x = x.view(x.size(0), -1) 
        x = self.dropout(F.relu(self.fc(x)))
        return x

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads=8, num_layers=4, dropout=0.1):
        super(DecoderTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # Transformer Decoder Layer
        # We use a standard TransformerDecoder to replace the LSTM
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_size, 
            nhead=num_heads, 
            dim_feedforward=embed_size * 4, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, features, captions):
        """
        features: (batch, embed_size) - from CNN
        captions: (batch, seq_len)
        """
        # Embed the captions (excluding <end> token)
        # captions: [batch, seq_len-1] -> [batch, seq_len-1, embed_size]
        tgt = self.embedding(captions[:, :-1])
               
        # In Transformers, we treat the image feature as the 'memory' (encoder output)
        # Transforms expect a rank 3 tensor, so we need to add an aaddtional
        # dummy dimension : (batch, 1, embed_size)
        memory = features.unsqueeze(1)
        
        # Create a causal mask for the captions (to prevent looking at the future)
        # This stops the Transformer from "cheating" by "looking" at the word it is supposed
        # to be predicting. Do
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        # Pass through the Transformer Decoder
        # The decoder uses caption embeddings as 'tgt' and image features as 'memory'
        output = self.transformer_decoder(tgt, memory, tgt_mask=tgt_mask)
        
        return self.linear(output)

## Updated Training Loop:

In [ ]:
from tqdm import tqdm 

IMAGE_SIZE = 224
BATCH_SIZE = 32
EMBED_SIZE = 256  # This acts as d_model for the Transformer

NUM_EPOCHS = 10
LEARNING_RATE = 1e-4 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab = train_dataset.vocab
VOCAB_SIZE = len(vocab)
PAD_IDX = vocab.stoi["<PAD>"]

# Initialize updated models
encoder = EncoderCNN(EMBED_SIZE).to(DEVICE)
# Use the DecoderTransformer we defined in the previous step
decoder = DecoderTransformer(EMBED_SIZE, VOCAB_SIZE).to(DEVICE)

# Initialize Loss and Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.Adam(params, lr=LEARNING_RATE)

for epoch in range(1, NUM_EPOCHS + 1):
    # --- TRAINING PHASE ---
    encoder.train()
    decoder.train()
    train_loss = 0
    
    loop = tqdm(train_loader, total=len(train_loader), leave=True)
    for images, captions in loop:
        images, captions = images.to(DEVICE), captions.to(DEVICE)

        optimizer.zero_grad()
        
        # 1. Forward pass
        features = encoder(images)
        # Note: DecoderTransformer handles the causal masking internally 
        # or you can pass a mask here.
        outputs = decoder(features, captions)

        # 2. Calculate loss
        # Transformer output is (Batch, Seq_Len-1, Vocab_Size) 
        # because the last token isn't predicted and the first is <SOS>.
        # Captions should be shifted: we predict tokens 1 to End given 0 to End-1.
        targets = captions[:, 1:] # Target is everything after <SOS>
        
        loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.reshape(-1))
        
        # 3. Backward pass
        loss.backward()
        # Optional: Add gradient clipping (Task 3 in your original assignment)
        torch.nn.utils.clip_grad_norm_(params, max_norm=5)
        optimizer.step()
        
        train_loss += loss.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    # --- VALIDATION PHASE ---
    encoder.eval()
    decoder.eval()
    val_loss = 0
    
    with torch.no_grad():
        for images, captions in val_loader:
            images, captions = images.to(DEVICE), captions.to(DEVICE)
            
            features = encoder(images)
            outputs = decoder(features, captions)
            
            targets = captions[:, 1:]
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.reshape(-1))
            val_loss += loss.item()

    # Log statistics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"\n=> Epoch {epoch} Complete")
    print(f"   Avg Train Loss: {avg_train_loss:.4f}")
    print(f"   Avg Val Loss:   {avg_val_loss:.4f}")

Epoch [7/10]: 100%|██████████| 938/938 [01:30<00:00, 10.37it/s, loss=2.51]



=> Epoch 7 Complete
   Avg Train Loss: 2.5150
   Avg Val Loss:   2.3471


Epoch [10/10]: 100%|██████████| 938/938 [01:29<00:00, 10.46it/s, loss=2.22]



=> Epoch 10 Complete
   Avg Train Loss: 2.2976
   Avg Val Loss:   2.1452


## Evaluation

In [ ]:
def generate_caption(encoder, decoder, image, vocab, max_length=20):
    encoder.eval()
    decoder.eval()
    
    result_caption = []
    
    with torch.no_grad():
        # 1. Process Image
        # image shape: [3, 224, 224] -> [1, 3, 224, 224]
        x = image.unsqueeze(0).to(DEVICE)
        features = encoder(x) # [1, embed_size]
        # Memory for transformer is the image feature
        memory = features.unsqueeze(1) # [1, 1, embed_size]
        
        # 2. Initialize sequence with <SOS> token
        sos_idx = vocab.stoi["<SOS>"]
        eos_idx = vocab.stoi["<EOS>"]
        
        # current_indices keeps track of the sequence generated so far
        current_indices = torch.tensor([[sos_idx]], device=DEVICE) # Shape: [1, 1]
        
        for _ in range(max_length):
            # Embed the sequence generated so far
            tgt = decoder.embedding(current_indices) # Shape: [1, seq_len, embed_size]
            
            # Create a causal mask for the current length
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(current_indices.size(1)).to(DEVICE)
            
            # Pass to Transformer Decoder
            # memory is the static image feature; tgt is the growing caption
            output = decoder.transformer_decoder(tgt, memory, tgt_mask=tgt_mask)
            
            # Predict the NEXT word using the last position of the output
            # output shape: [1, seq_len, embed_size] -> take [:, -1, :]
            logits = decoder.linear(output[:, -1, :]) # Shape: [1, vocab_size]
            predicted = logits.argmax(1)
            
            # Update the sequence for the next iteration
            current_indices = torch.cat((current_indices, predicted.unsqueeze(0)), dim=1)
            
            token = vocab.itos[predicted.item()]
            if token == "<EOS>":
                break
            
            result_caption.append(token)
            
    return ' '.join(result_caption)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Switch models to evaluation mode
encoder.eval()
decoder.eval()

# 2. Get a random batch from validation set
images, captions = next(iter(val_loader))
sample_img = images[0]
sample_cap = captions[0]

# 4. Generate prediction using the updated function
prediction = generate_caption(encoder, decoder, sample_img, vocab)

# 5. Convert ground truth (cleaning up special tokens)
ground_truth = [vocab.itos[idx.item()] for idx in sample_cap 
                if vocab.itos[idx.item()] not in ["<PAD>", "<SOS>", "<EOS>"]]
ground_truth = ' '.join(ground_truth)

# 6. Plotting with Un-normalization
img_display = sample_img.permute(1, 2, 0).cpu().numpy()

# Undo normalisation
mean = np.array([0.485, 0.4461, 0.4039]) 
std = np.array([0.2695, 0.2623, 0.2772])

img_display = std * img_display + mean
img_display = np.clip(img_display, 0, 1)

plt.figure(figsize=(10, 6))
plt.imshow(img_display)
plt.title(f"Predicted: {prediction}\nActual: {ground_truth}", fontsize=12)
plt.axis('off')
plt.show()

## Exercises


1. Scaled Attention. Create a string task_3_explanation. Discuss why we divide the dot product by $\sqrt{d_k}$. What would happen to the Softmax gradients if $d_k$ was very large and we didn't scale?

2. Batch Size, `d_model` & Attention Heads. Experiment with num_heads (e.g., 4 vs 8), `d_model` (e.g. 8,16,24,64,128,512) and batch sizes. Discuss the impact on training time and caption quality in task_4_explanation.

3. Regularisation. Layer Normalisation is the most common form of Normalization applied to Transformer models. Adjust the level of normalisation in the model by adjusting the `layer_norm_eps` parameter of nn.Transformer, as well as the level dropout. (https://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.htmlhttps://docs.pytorch.org/docs/stable/generated/torch.nn.Transformer.html). Explain your understanding of the role of LayerNorm in transformers, and your observations over your results in a string task_5_explanation

4. Compare the performance of this model against your CNN/LST model from the last assignement. Note any observations in terms of model size, accuracy, training time, and/or quality of captions in a string task_5_explanation



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Exercise 1 – Why do we scale by sqrt(d_k)?
# ═══════════════════════════════════════════════════════════════════
task_3_explanation = """
The attention score between a query and a key is their dot product, the sum of
d_k individual multiplications. As d_k grows, the expected magnitude of this sum
also grows, roughly proportional to sqrt(d_k) for randomly initialised unit-variance
vectors.

If d_k is large and we do not scale, the dot products become very large in magnitude.
When passed into softmax, large values cause the output to become extremely peaked, 
one attention weight approaches 1.0 and all others approach 0.0. This is softmax
saturation. In the saturated region the gradient of softmax is near zero, causing
vanishing gradients during backpropagation and the model receives almost no learning
signal through the attention weights and training stalls.

Dividing by sqrt(d_k) keeps the dot products in a moderate range regardless of
embedding dimension, ensuring the softmax output remains a smooth distribution
where gradients can flow properly during training.
"""
print(task_3_explanation)


# ═══════════════════════════════════════════════════════════════════
# Exercise 2 – Effect of num_heads, d_model, and batch size
# ═══════════════════════════════════════════════════════════════════
task_4_explanation = """
d_model (Embed Size):
  Embed sizes from 8 to 1024 were tested. A larger d_model gives each attention head 
  more representational capacity and the feedforward sublayers wider internal
  representations, while very small d_model values cause severe underfitting. Loss
  improved consistently and substantially with each increase in d_model. Training
  time was relatively stable from d_model sizes 8 to 256, with a noticeable increase
  with embed size 512, and a very large increase in training time with embed size 1024.

num_heads:
  At every d_model tested, num_heads had minimal effect on val loss. More heads
  allow the model to attend to different representation subspaces in parallel,
  but the benefit is marginal here. The dominant limiting factor is d_model and
  the spatially compressed CNN encoder, not the number of attention heads. Training
  time was also nearly identical across head counts at the same d_model.

Batch size:
  Smaller batch sizes (8, 16) gave lower val loss but significantly longer training
  time. Larger batches (64, 128) trained faster but with worse loss, likely because
  smaller batches produce noisier gradient estimates which may help the model avoid
  poor local optima.

For the following exercise batch=32, d_model=512, num_heads=8 were chosen, balancing
training time and performance.
"""
print(task_4_explanation)


# ═══════════════════════════════════════════════════════════════════
# Exercise 3 – LayerNorm and dropout regularisation
# ═══════════════════════════════════════════════════════════════════
task_5_explanation = """
All experiments used batch=32, d_model=512, num_heads=8.

Role of Layer Normalisation in Transformers:
  Layer Normalisation normalises activations across the feature dimension for
  each individual sample, computing mean and variance over the d_model dimensions
  at each position. This is in contrast to Batch Normalisation which normalises
  across the batch dimension. LayerNorm is preferred in Transformers because it
  is independent of batch size, stable with variable sequence lengths, and
  prevents activations from growing uncontrollably through many layers via the
  residual connections.

layer_norm_eps:
  This is a small constant added to the variance for numerical stability,
  preventing division by zero. Varying it across several orders of magnitude
  (1e-6 to 1e-4) with 0.1 dropout produced almost identical results. Setting 
  layernorm to 0 also had negligible effect. This confirms that layer_norm_eps
  is a numerical stability parameter rather than a meaningful regularisation
  lever. Varying it across several orders of magnitude produced very similar
  results, confirming the model is insensitive to it within a reasonable range. 
  The best model in this exercise however did have layernorm = 1e-5 with no dropout.

Dropout:
  Dropout had a much larger effect than layer_norm_eps. Removing dropout
  entirely gave the best val loss and the best caption quality observed across
  all experiments. This is consistent with Assignment 3 findings. The model is
  not overfitting on Flickr8k, so dropout only introduces unnecessary noise that
  interferes with learning. Increasing dropout to 0.3 and 0.5 progressively
  worsened performance, the same pattern seen with the LSTM decoder in Assignment 3, 
  though to an even larger extent here.
"""
print(task_5_explanation)


# ═══════════════════════════════════════════════════════════════════
# Exercise 4 – CNN+Transformer vs CNN+LSTM comparison
# ═══════════════════════════════════════════════════════════════════
task_6_explanation = """
Val loss:
  Loss values between comparable LSTM and Transformer models were relatively similar, 
  if not better for the LSTM models. 

Caption quality:
  Despite similar val losses, the Transformer produced noticeably better
  captions for d_model >= 512.
  The LSTM collapsed to a generic sentence for all images across all
  configurations. This illustrates a limitation of cross-entropy val loss
  as an evaluation metric, a lower loss does not necessarily mean better
  caption quality.

Training time:
  The Transformer took noticeably longer to train compared to the LSTM at
  comparable embed/hidden sizes. The Transformer is a significantly larger
  model due to its multiple attention layers, feedforward sublayers, and layer
  normalisation across 4 decoder layers. Multi-head attention computes O(n^2)
  attention matrices, while the LSTM processes tokens sequentially but has a
  simpler per-step computation.

Model size:
  The Transformer decoder has considerably more parameters than the single-layer
  LSTM decoder, particularly at larger d_model values. This increased capacity
  is what allows it to produce more varied captions despite the same spatially
  compressed CNN encoder.

Summary
  For image captioning on Flickr8k, the Transformer decoder outperforms the
  LSTM decoder in caption quality at the cost of larger model size and
  slower training. 
"""
print(task_6_explanation)
